In [1]:
import polars as pl
import pandas as pd
import numpy as np
print("Polars version:", pl.__version__)
print("Pandas version:", pd.__version__)
print("NumPy version:", np.__version__)
print("✅ All imports working!")

Polars version: 1.44.1
Pandas version: 3.0.5
NumPy version: 2.4.6
✅ All imports working!


In [2]:
# Define dataset path
DATA_PATH = r"D:\FYP\datasets\US\US_Accidents_March23.csv"

# Load lazily - does NOT load into RAM yet
df_lazy = pl.scan_csv(DATA_PATH)

# Check shape (rows x columns) - this WILL scan the file, takes ~1-2 min
shape = df_lazy.select(pl.len()).collect()
print(f"Total rows: {shape[0,0]:,}")

# Check column names and data types instantly (no scanning needed)
print(f"\nTotal columns: {len(df_lazy.columns)}")
print("\nColumn names and types:")
for col, dtype in zip(df_lazy.columns, df_lazy.dtypes):
    print(f"  {col:35s} → {dtype}")

Total rows: 7,728,394

Total columns: 46

Column names and types:
  ID                                  → String
  Source                              → String
  Severity                            → Int64
  Start_Time                          → String
  End_Time                            → String
  Start_Lat                           → Float64
  Start_Lng                           → Float64
  End_Lat                             → String
  End_Lng                             → String
  Distance(mi)                        → Float64
  Description                         → String
  Street                              → String
  City                                → String
  County                              → String
  State                               → String
  Zipcode                             → String
  Country                             → String
  Timezone                            → String
  Airport_Code                        → String
  Weather_Timestamp                   →

C:\Users\Admin\AppData\Local\Temp\ipykernel_11384\234065437.py:12: PerformanceWarning: Determining the column names of a LazyFrame requires resolving its schema, which is a potentially expensive operation. Use `LazyFrame.collect_schema().names()` to get the column names without this warning.
  print(f"\nTotal columns: {len(df_lazy.columns)}")
C:\Users\Admin\AppData\Local\Temp\ipykernel_11384\234065437.py:14: PerformanceWarning: Determining the column names of a LazyFrame requires resolving its schema, which is a potentially expensive operation. Use `LazyFrame.collect_schema().names()` to get the column names without this warning.
  for col, dtype in zip(df_lazy.columns, df_lazy.dtypes):
C:\Users\Admin\AppData\Local\Temp\ipykernel_11384\234065437.py:14: PerformanceWarning: Determining the data types of a LazyFrame requires resolving its schema, which is a potentially expensive operation. Use `LazyFrame.collect_schema().dtypes()` to get the data types without this warning.
  for col, dty

In [3]:
# Fix: use collect_schema() - no more warnings
schema = df_lazy.collect_schema()

print(f"Total columns: {len(schema.names())}")
print("\nAll 46 Columns and Types:")
print("-" * 50)
for col, dtype in zip(schema.names(), schema.dtypes()):
    print(f"  {col:35s} → {dtype}")

Total columns: 46

All 46 Columns and Types:
--------------------------------------------------
  ID                                  → String
  Source                              → String
  Severity                            → Int64
  Start_Time                          → String
  End_Time                            → String
  Start_Lat                           → Float64
  Start_Lng                           → Float64
  End_Lat                             → String
  End_Lng                             → String
  Distance(mi)                        → Float64
  Description                         → String
  Street                              → String
  City                                → String
  County                              → String
  State                               → String
  Zipcode                             → String
  Country                             → String
  Timezone                            → String
  Airport_Code                        → String
  Weather

In [4]:
# Count nulls per column - scans the full file (~1-2 min)
print("Calculating null counts... please wait")

null_counts = (
    df_lazy
    .select([
        pl.col(c).is_null().sum().alias(c)
        for c in schema.names()
    ])
    .collect()
)

total_rows = 7_728_394

print(f"\n{'Column':<35} {'Null Count':>12} {'Null %':>8}")
print("-" * 58)
for col in schema.names():
    count = null_counts[col][0]
    pct = (count / total_rows) * 100
    flag = " ⚠️" if pct > 20 else ""
    print(f"  {col:<33} {count:>12,} {pct:>7.2f}%{flag}")

Calculating null counts... please wait

Column                                Null Count   Null %
----------------------------------------------------------
  ID                                           0    0.00%
  Source                                       0    0.00%
  Severity                                     0    0.00%
  Start_Time                                   0    0.00%
  End_Time                                     0    0.00%
  Start_Lat                                    0    0.00%
  Start_Lng                                    0    0.00%
  End_Lat                              3,402,762   44.03% ⚠️
  End_Lng                              3,402,762   44.03% ⚠️
  Distance(mi)                                 0    0.00%
  Description                                  5    0.00%
  Street                                  10,869    0.14%
  City                                       253    0.00%
  County                                       0    0.00%
  State                  

In [5]:
# Full null analysis - sorted by worst nulls first
print(f"{'Column':<35} {'Null Count':>12} {'Null %':>8}  Status")
print("-" * 72)

results = []
for col in schema.names():
    count = null_counts[col][0]
    pct = (count / total_rows) * 100
    results.append((col, count, pct))

# Sort by null % descending
results.sort(key=lambda x: x[2], reverse=True)

for col, count, pct in results:
    if pct > 20:
        status = "❌ DROP (already planned)"
    elif pct > 5:
        status = "⚠️  HIGH - needs imputation"
    elif pct > 0:
        status = "🔶 LOW  - needs imputation"
    else:
        status = "✅ Clean"
    print(f"  {col:<33} {count:>12,} {pct:>7.2f}%  {status}")

Column                                Null Count   Null %  Status
------------------------------------------------------------------------
  End_Lat                              3,402,762   44.03%  ❌ DROP (already planned)
  End_Lng                              3,402,762   44.03%  ❌ DROP (already planned)
  Precipitation(in)                    2,203,586   28.51%  ❌ DROP (already planned)
  Wind_Chill(F)                        1,999,019   25.87%  ❌ DROP (already planned)
  Wind_Speed(mph)                        571,233    7.39%  ⚠️  HIGH - needs imputation
  Visibility(mi)                         177,098    2.29%  🔶 LOW  - needs imputation
  Wind_Direction                         175,206    2.27%  🔶 LOW  - needs imputation
  Humidity(%)                            174,144    2.25%  🔶 LOW  - needs imputation
  Weather_Condition                      173,459    2.24%  🔶 LOW  - needs imputation
  Temperature(F)                         163,853    2.12%  🔶 LOW  - needs imputation
  Pressure(in

In [6]:
# Check Severity class distribution
severity_dist = (
    df_lazy
    .group_by("Severity")
    .agg(pl.len().alias("Count"))
    .sort("Severity")
    .collect()
)

print("Severity Distribution:")
print("-" * 45)
total = severity_dist["Count"].sum()
for row in severity_dist.iter_rows(named=True):
    pct = (row["Count"] / total) * 100
    bar = "█" * int(pct / 2)
    print(f"  Severity {row['Severity']}:  {row['Count']:>9,}  ({pct:5.2f}%)  {bar}")

print(f"\n  {'TOTAL':>10}  {total:>9,}  (100.00%)")

Severity Distribution:
---------------------------------------------
  Severity 1:     67,366  ( 0.87%)  
  Severity 2:  6,156,981  (79.67%)  ███████████████████████████████████████
  Severity 3:  1,299,337  (16.81%)  ████████
  Severity 4:    204,710  ( 2.65%)  █

       TOTAL  7,728,394  (100.00%)


In [7]:
# Final list of columns to DROP
# Original plan + Precipitation & Wind_Chill (too many nulls)
cols_to_drop = [
    "ID",
    "Country",
    "Airport_Code",
    "Weather_Timestamp",
    "Zipcode",
    "Description",
    "End_Lat",
    "End_Lng",
    "Street",
    "Precipitation(in)",   # 28.51% nulls - dropping
    "Wind_Chill(F)",       # 25.87% nulls - dropping
]

df_clean = df_lazy.drop(cols_to_drop)

# Verify remaining columns
remaining = df_clean.collect_schema().names()
print(f"Columns remaining: {len(remaining)}  (was 46)")
print("\nRemaining columns:")
for i, col in enumerate(remaining, 1):
    print(f"  {i:2}. {col}")

Columns remaining: 35  (was 46)

Remaining columns:
   1. Source
   2. Severity
   3. Start_Time
   4. End_Time
   5. Start_Lat
   6. Start_Lng
   7. Distance(mi)
   8. City
   9. County
  10. State
  11. Timezone
  12. Temperature(F)
  13. Humidity(%)
  14. Pressure(in)
  15. Visibility(mi)
  16. Wind_Direction
  17. Wind_Speed(mph)
  18. Weather_Condition
  19. Amenity
  20. Bump
  21. Crossing
  22. Give_Way
  23. Junction
  24. No_Exit
  25. Railway
  26. Roundabout
  27. Station
  28. Stop
  29. Traffic_Calming
  30. Traffic_Signal
  31. Turning_Loop
  32. Sunrise_Sunset
  33. Civil_Twilight
  34. Nautical_Twilight
  35. Astronomical_Twilight


In [8]:
# --- Numerical columns: fill with MEDIAN ---
numerical_to_impute = [
    "Temperature(F)",
    "Humidity(%)",
    "Pressure(in)",
    "Visibility(mi)",
    "Wind_Speed(mph)",
]

# --- Categorical columns: fill with "Unknown" ---
categorical_to_impute = [
    "Wind_Direction",
    "Weather_Condition",
    "Timezone",
    "City",
    "Sunrise_Sunset",
    "Civil_Twilight",
    "Nautical_Twilight",
    "Astronomical_Twilight",
    "Source",
]

# Step 1: Compute medians first (fast, no full scan needed)
medians = (
    df_clean
    .select([pl.col(c).median().alias(c) for c in numerical_to_impute])
    .collect()
)

print("Median values for imputation:")
for col in numerical_to_impute:
    print(f"  {col:<20} → {medians[col][0]:.4f}")

Median values for imputation:
  Temperature(F)       → 64.0000
  Humidity(%)          → 67.0000
  Pressure(in)         → 29.8600
  Visibility(mi)       → 10.0000
  Wind_Speed(mph)      → 7.0000


In [9]:
# Apply median imputation to numerical columns
df_clean = df_clean.with_columns([
    pl.col("Temperature(F)").fill_null(medians["Temperature(F)"][0]),
    pl.col("Humidity(%)").fill_null(medians["Humidity(%)"][0]),
    pl.col("Pressure(in)").fill_null(medians["Pressure(in)"][0]),
    pl.col("Visibility(mi)").fill_null(medians["Visibility(mi)"][0]),
    pl.col("Wind_Speed(mph)").fill_null(medians["Wind_Speed(mph)"][0]),
])

# Apply "Unknown" fill to categorical columns
df_clean = df_clean.with_columns([
    pl.col(c).fill_null("Unknown") for c in categorical_to_impute
])

# Collect into memory (this is the big step - takes 2-4 min)
print("Loading cleaned dataset into memory... please wait ⏳")
df = df_clean.collect()

# Remove duplicates
before = len(df)
df = df.unique()
after = len(df)

print(f"\n✅ Dataset loaded!")
print(f"   Rows before dedup : {before:,}")
print(f"   Rows after dedup  : {after:,}")
print(f"   Duplicates removed: {before - after:,}")
print(f"\n   Memory usage: {df.estimated_size('mb'):.1f} MB")

Loading cleaned dataset into memory... please wait ⏳

✅ Dataset loaded!
   Rows before dedup : 7,728,394
   Rows after dedup  : 7,606,833
   Duplicates removed: 121,561

   Memory usage: 1261.8 MB


In [10]:
# First check what Start_Time looks like (it's still a String)
print("Sample Start_Time values:")
print(df["Start_Time"].head(5))

Sample Start_Time values:
shape: (5,)
Series: 'Start_Time' [str]
[
	"2020-11-17 07:53:01"
	"2019-12-04 06:51:18"
	"2018-10-05 16:14:57"
	"2017-11-24 18:45:22"
	"2020-12-07 08:32:57"
]


In [11]:
# Parse datetime strings → actual datetime type
df = df.with_columns([
    pl.col("Start_Time").str.to_datetime(format="%Y-%m-%d %H:%M:%S", strict=False),
    pl.col("End_Time").str.to_datetime(format="%Y-%m-%d %H:%M:%S", strict=False),
])

# Extract features from Start_Time
df = df.with_columns([
    pl.col("Start_Time").dt.hour().alias("Hour"),
    pl.col("Start_Time").dt.weekday().alias("Day_of_Week"),   # 0=Mon, 6=Sun
    pl.col("Start_Time").dt.month().alias("Month"),
    pl.col("Start_Time").dt.year().alias("Year"),
])

# Compute accident duration in minutes
df = df.with_columns([
    ((pl.col("End_Time") - pl.col("Start_Time"))
     .dt.total_minutes()
     .alias("Accident_Duration_Minutes"))
])

# Boolean flags
df = df.with_columns([
    (pl.col("Day_of_Week") >= 5).alias("Is_Weekend"),
    (
        ((pl.col("Hour") >= 7) & (pl.col("Hour") <= 9)) |
        ((pl.col("Hour") >= 16) & (pl.col("Hour") <= 19))
    ).alias("Is_Rush_Hour"),
])

# Drop raw datetime columns (no longer needed)
df = df.drop(["Start_Time", "End_Time"])

# Verify new features
print("✅ New features created!")
print(f"\nTotal columns now: {len(df.columns)}")
print("\nSample of new features (first 5 rows):")
print(df.select([
    "Hour", "Day_of_Week", "Month", "Year",
    "Accident_Duration_Minutes", "Is_Weekend", "Is_Rush_Hour"
]).head(5))

✅ New features created!

Total columns now: 40

Sample of new features (first 5 rows):
shape: (5, 7)
┌──────┬─────────────┬───────┬──────┬───────────────────────────┬────────────┬──────────────┐
│ Hour ┆ Day_of_Week ┆ Month ┆ Year ┆ Accident_Duration_Minutes ┆ Is_Weekend ┆ Is_Rush_Hour │
│ ---  ┆ ---         ┆ ---   ┆ ---  ┆ ---                       ┆ ---        ┆ ---          │
│ i8   ┆ i8          ┆ i8    ┆ i32  ┆ i64                       ┆ bool       ┆ bool         │
╞══════╪═════════════╪═══════╪══════╪═══════════════════════════╪════════════╪══════════════╡
│ 7    ┆ 2           ┆ 11    ┆ 2020 ┆ 75                        ┆ false      ┆ true         │
│ 6    ┆ 3           ┆ 12    ┆ 2019 ┆ 240                       ┆ false      ┆ false        │
│ 16   ┆ 5           ┆ 10    ┆ 2018 ┆ 29                        ┆ true       ┆ true         │
│ 18   ┆ 5           ┆ 11    ┆ 2017 ┆ 360                       ┆ true       ┆ true         │
│ 8    ┆ 1           ┆ 12    ┆ 2020 ┆ 46             

In [12]:
duration_stats = df.select("Accident_Duration_Minutes").describe()
print("Duration Statistics:")
print(duration_stats)

# Check for negative or extreme values
bad = df.filter(pl.col("Accident_Duration_Minutes") <= 0)
extreme = df.filter(pl.col("Accident_Duration_Minutes") > 1440)  # > 24 hours

print(f"\nNegative/zero duration rows : {len(bad):,}")
print(f"Extreme duration (>24h) rows: {len(extreme):,}")

Duration Statistics:
shape: (9, 2)
┌────────────┬───────────────────────────┐
│ statistic  ┆ Accident_Duration_Minutes │
│ ---        ┆ ---                       │
│ str        ┆ f64                       │
╞════════════╪═══════════════════════════╡
│ count      ┆ 6.866812e6                │
│ null_count ┆ 740021.0                  │
│ mean       ┆ 382.137065                │
│ std        ┆ 12373.856908              │
│ min        ┆ 1.0                       │
│ 25%        ┆ 30.0                      │
│ 50%        ┆ 61.0                      │
│ 75%        ┆ 122.0                     │
│ max        ┆ 2.812939e6                │
└────────────┴───────────────────────────┘

Negative/zero duration rows : 0
Extreme duration (>24h) rows: 26,275


In [13]:
# Fix 1: Cap anything above 24 hours (1440 min) → set to 1440
# Fix 2: Fill nulls with median duration (61 minutes)

df = df.with_columns([
    pl.col("Accident_Duration_Minutes")
    .clip(upper_bound=1440)          # cap at 24 hours
    .fill_null(61)                   # fill nulls with median
    .alias("Accident_Duration_Minutes")
])

# Verify fixes
stats_after = df.select("Accident_Duration_Minutes").describe()
print("Duration after fixing:")
print(stats_after)

Duration after fixing:
shape: (9, 2)
┌────────────┬───────────────────────────┐
│ statistic  ┆ Accident_Duration_Minutes │
│ ---        ┆ ---                       │
│ str        ┆ f64                       │
╞════════════╪═══════════════════════════╡
│ count      ┆ 7.606833e6                │
│ null_count ┆ 0.0                       │
│ mean       ┆ 104.890335                │
│ std        ┆ 140.602291                │
│ min        ┆ 1.0                       │
│ 25%        ┆ 33.0                      │
│ 50%        ┆ 61.0                      │
│ 75%        ┆ 113.0                     │
│ max        ┆ 1440.0                    │
└────────────┴───────────────────────────┘


In [14]:
# Categorical columns to label-encode
# Tree-based models (RF, XGBoost, LightGBM) work perfectly with label encoding
cat_cols = [
    "Source", "City", "County", "State", "Timezone",
    "Wind_Direction", "Weather_Condition",
    "Sunrise_Sunset", "Civil_Twilight",
    "Nautical_Twilight", "Astronomical_Twilight",
]

# Convert to Categorical then to integer codes
df = df.with_columns([
    pl.col(c).cast(pl.Categorical).to_physical().alias(c)
    for c in cat_cols
])

# Convert boolean columns to integers (0/1) for ML compatibility
bool_cols = [
    "Amenity", "Bump", "Crossing", "Give_Way", "Junction",
    "No_Exit", "Railway", "Roundabout", "Station", "Stop",
    "Traffic_Calming", "Traffic_Signal", "Turning_Loop",
    "Is_Weekend", "Is_Rush_Hour",
]

df = df.with_columns([
    pl.col(c).cast(pl.Int8) for c in bool_cols
])

print("✅ Encoding complete!")
print(f"\nFinal dataset shape: {df.shape[0]:,} rows × {df.shape[1]} columns")
print("\nFinal column dtypes:")
schema = df.schema
for col, dtype in schema.items():
    print(f"  {col:<35} → {dtype}")

✅ Encoding complete!

Final dataset shape: 7,606,833 rows × 40 columns

Final column dtypes:
  Source                              → UInt32
  Severity                            → Int64
  Start_Lat                           → Float64
  Start_Lng                           → Float64
  Distance(mi)                        → Float64
  City                                → UInt32
  County                              → UInt32
  State                               → UInt32
  Timezone                            → UInt32
  Temperature(F)                      → Float64
  Humidity(%)                         → Float64
  Pressure(in)                        → Float64
  Visibility(mi)                      → Float64
  Wind_Direction                      → UInt32
  Wind_Speed(mph)                     → Float64
  Weather_Condition                   → UInt32
  Amenity                             → Int8
  Bump                                → Int8
  Crossing                            → Int8
  Give_Way   

In [15]:
# Final null check - confirm no remaining nulls
remaining_nulls = df.null_count()
total_nulls = remaining_nulls.to_numpy().sum()
print(f"Total remaining nulls: {total_nulls}")

if total_nulls == 0:
    print("✅ Zero nulls — dataset is perfectly clean!")
else:
    print("⚠️  Remaining nulls found:")
    for col in df.columns:
        n = df[col].null_count()
        if n > 0:
            print(f"   {col}: {n:,}")

Total remaining nulls: 4440126
⚠️  Remaining nulls found:
   Hour: 740,021
   Day_of_Week: 740,021
   Month: 740,021
   Year: 740,021
   Is_Weekend: 740,021
   Is_Rush_Hour: 740,021


In [16]:
import os

# Save cleaned dataset as parquet (compressed)
save_path = r"D:\FYP\SafeRoute-AI\data\us_accidents_clean.parquet"

print("Saving to parquet... ⏳")
df.write_parquet(save_path, compression="snappy")

# Verify file was saved
size_mb = os.path.getsize(save_path) / (1024 * 1024)
print(f"✅ Saved successfully!")
print(f"   Path : {save_path}")
print(f"   Size : {size_mb:.1f} MB  (was 3,058 MB as CSV)")
print(f"   Compression ratio: {3058 / size_mb:.1f}x smaller!")

Saving to parquet... ⏳
✅ Saved successfully!
   Path : D:\FYP\SafeRoute-AI\data\us_accidents_clean.parquet
   Size : 302.6 MB  (was 3,058 MB as CSV)
   Compression ratio: 10.1x smaller!


In [17]:
# Compute median values for datetime-derived columns
hour_median    = int(df["Hour"].drop_nulls().median())
dow_median     = int(df["Day_of_Week"].drop_nulls().median())
month_median   = int(df["Month"].drop_nulls().median())
year_median    = int(df["Year"].drop_nulls().median())

print("Fill values:")
print(f"  Hour        → {hour_median}")
print(f"  Day_of_Week → {dow_median}")
print(f"  Month       → {month_median}")
print(f"  Year        → {year_median}")

# Fill nulls
df = df.with_columns([
    pl.col("Hour").fill_null(hour_median),
    pl.col("Day_of_Week").fill_null(dow_median),
    pl.col("Month").fill_null(month_median),
    pl.col("Year").fill_null(year_median),
    pl.col("Is_Weekend").fill_null(0),    # default: weekday
    pl.col("Is_Rush_Hour").fill_null(0),  # default: not rush hour
])

# Confirm zero nulls
total_nulls = df.null_count().to_numpy().sum()
print(f"\nTotal nulls remaining: {total_nulls}")
print("✅ Dataset is 100% clean!" if total_nulls == 0 else "❌ Still has nulls!")

Fill values:
  Hour        → 13
  Day_of_Week → 4
  Month       → 7
  Year        → 2020

Total nulls remaining: 0
✅ Dataset is 100% clean!


In [18]:
# Re-save the corrected parquet
print("Re-saving corrected parquet... ⏳")
df.write_parquet(save_path, compression="snappy")
print(f"✅ Parquet updated at: {save_path}")
print(f"   Final shape: {df.shape[0]:,} rows × {df.shape[1]} columns")

Re-saving corrected parquet... ⏳
✅ Parquet updated at: D:\FYP\SafeRoute-AI\data\us_accidents_clean.parquet
   Final shape: 7,606,833 rows × 40 columns
